# SMART Checkpoint Evaluation (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-constrained/notebooks/smart_checkpoint_eval_colab.ipynb)

This notebook is the **evaluation stage** only:
- load simulation manifests,
- bind metrics JSON to manifests,
- run strict contract checks and write eval artifact.


In [ ]:
# Step 1: Sync repo + deterministic Colab bootstrap
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for path in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


In [ ]:
# Step 2: Load config + locate manifests and metrics
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config

EXPERIMENT_SLUG = "smart-constrained"
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root=".")
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root=".")
display(bundle.summary_table)

RUN = dict(cfg.get("run", {}))
SMART = dict(cfg.get("smart", {}))
CONSTRAINTS = dict(cfg.get("constraints", {}))
EVAL = dict(cfg.get("evaluation", {}))

RUN_NAME = str(RUN.get("run_name", "dev"))
RUN_PREFIX = str(RUN.get("run_prefix", "smart_constrained"))
PERSIST_ROOT = str(RUN.get("persist_root", "/content/drive/MyDrive/wosac_experiments"))
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode("utf-8")).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f"{RUN_PREFIX}_{RUN_NAME}"
persist_run_dir.mkdir(parents=True, exist_ok=True)

SIM_MANIFESTS_DIR = Path(
    os.environ.get("WOSAC_SIM_MANIFESTS_DIR", str(persist_run_dir / "outputs" / "simulation_manifests"))
).expanduser()
METRICS_DIR = Path(os.environ.get("WOSAC_MODEL_METRICS_DIR", "").strip()).expanduser()

assert str(METRICS_DIR), "Set WOSAC_MODEL_METRICS_DIR before running eval notebook"
assert METRICS_DIR.exists(), f"Metrics dir does not exist: {METRICS_DIR}"

print("RUN_TAG:", RUN_TAG)
print("SIM_MANIFESTS_DIR:", SIM_MANIFESTS_DIR)
print("METRICS_DIR:", METRICS_DIR)
print("cfg_hash:", cfg_hash)


In [ ]:
# Step 3: Build model list from simulation manifests
from src.workflows import load_simulation_manifest

manifest_paths = sorted(SIM_MANIFESTS_DIR.glob('*_simulation_manifest.json'))
assert manifest_paths, f"No manifests found in {SIM_MANIFESTS_DIR}"

models = []
manifest_load_rows = []
for manifest_path in manifest_paths:
    manifest, errors = load_simulation_manifest(manifest_path)
    model_id = str(manifest.get("model_id", "")).strip() or manifest_path.name.replace('_simulation_manifest.json', '')
    models.append({
        "model_id": model_id,
        "checkpoint_path": str(manifest.get("checkpoint_path", "")),
        "manifest_json": str(manifest_path),
        "metrics_json": str(METRICS_DIR / f"{model_id}.json"),
        "env": {},
    })
    manifest_load_rows.append({
        "model_id": model_id,
        "manifest_json": str(manifest_path),
        "manifest_errors": errors,
    })

print("num_models:", len(models))
for row in manifest_load_rows:
    print(row["model_id"], "errors=", row["manifest_errors"])


In [ ]:
# Step 4: Strict evaluation contract checks + metrics ingestion
from src.workflows import run_smart_eval_flow

flow_bundle = run_smart_eval_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    smart_repo_url=str(SMART.get("repo_url", "https://github.com/rainmaker22/SMART.git")),
    smart_repo_branch=str(SMART.get("branch", "main")),
    smart_repo_dir=str(SMART.get("repo_dir", "/content/SMART")),
    smart_val_config=str(SMART.get("val_config", "configs/validation/validation_scalable.yaml")),
    sync_smart_repo=False,
    models=models,
    strict_contract=True,
    require_metrics_binding=True,
    verify_checkpoint_hash=True,
    compatibility_keys=[
        "scenario_set_hash",
        "evaluator_id",
        "metrics_config_hash",
        "n_rollouts",
        "num_history_seconds",
        "num_future_seconds",
    ],
)

print("summary:")
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))


In [ ]:
# Step 5: Contract gate table (hard fail on invalid models)
import pandas as pd

rows = []
for model in flow_bundle.models:
    rows.append({
        "model_id": model.get("model_id"),
        "contract_valid": bool(model.get("contract_valid", False)),
        "contract_errors": ";".join(model.get("contract_errors", [])),
        "metrics_source": model.get("metrics_source"),
        "realism_meta_metric": (model.get("metrics", {}) or {}).get("realism_meta_metric"),
    })

df = pd.DataFrame(rows)
display(df)

invalid = [r for r in rows if not bool(r["contract_valid"]) ]
if invalid:
    raise RuntimeError(f"Strict contract failed for {len(invalid)} model(s). Check contract_errors in table.")

print("All models passed strict contract checks.")


In [ ]:
# Step 6: Write checkpoint eval artifact (Drive)
payload = {
    "run_id": "smart_checkpoint_eval_v0",
    "status": str(flow_bundle.summary.get("status", "unknown")),
    "run_tag": RUN_TAG,
    "primary_metric": str(EVAL.get("primary_metric", "realism_meta_metric")),
    "models": flow_bundle.models,
    "summary": flow_bundle.summary,
    "artifacts": flow_bundle.artifacts,
    "constraints": CONSTRAINTS,
    "notes": [
        "Evaluation-only notebook. Simulation manifests are produced by smart_rollout_simulation_colab.ipynb.",
    ],
    "provenance": {
        "config_hash": cfg_hash,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "experiment_slug": EXPERIMENT_SLUG,
    },
}

drive_path = persist_run_dir / "outputs" / "smart_checkpoint_eval_v0.json"
drive_path.parent.mkdir(parents=True, exist_ok=True)
drive_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("drive_path:", drive_path)
